<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 4 — Python Toolkit</div>
</div>

#### Masks and slices, reproducible random numbers, scipy.stats, and tables with one row per group.
***

## Why this notebook

Four pieces of Python come up in every lab from now on, and in Exercise 2. Fifteen minutes; skim what you know.

1. masks and slices: selecting from arrays without loops;
2. random numbers you can reproduce;
3. `scipy.stats`: distributions and tests;
4. pandas tables: one row per group.

Section 5 has optional practice: six short exercises on a small weather record, each with a number to check against. Practice is how this sticks: do them now if you're new to these tools, or come back to them this week.

## <span style="color:#B5121B">1. Masks and slices</span>

A comparison on an array gives an array of `True`/`False` (a **mask**), which selects elements. Slices shift an array against itself, which is how you compare neighbours.

In [ ]:
import numpy as np
x = np.array([3.1, 2.9, 3.0, 9.5, 3.2, 3.1, 0.2, 3.0])
big = x > 5
print(big, x[big], np.where(big)[0])      # the mask, the values, the positions
print(x[(x > 1) & (x < 5)])               # combine masks with & and |, with brackets
print(np.diff(x))                         # x[1:] - x[:-1]: differences of neighbours
print((x[2:] - x[:-2]) / 2)               # central differences (Week 3), no loop

In [ ]:
# Your turn: find the positions where x changes by more than 2 from one value to the next

## <span style="color:#B5121B">2. Random numbers you can reproduce</span>

Make **one** generator with a seed at the top of a notebook and draw everything from it. Run the notebook again and you get the same numbers; change the seed to see how much a result depends on chance.

In [ ]:
rng = np.random.default_rng(42)
print(rng.normal(10, 2, size=3))          # Gaussian: mean 10, standard deviation 2
print(rng.uniform(0, 1, size=3))
print(rng.choice([1, 2, 3, 4], size=4, replace=True))   # resampling with replacement (bootstrap)
print(rng.permutation(5))                 # a shuffle (permutation tests)

## <span style="color:#B5121B">3. scipy.stats</span>

Each distribution has the same methods: `pdf`, `cdf`, `sf` (= 1 − cdf, a tail probability), `ppf` (the inverse of the cdf). Tests return a result with `.statistic` and `.pvalue`.

In [ ]:
from scipy import stats
print(stats.norm.sf(2))                   # P(Z > 2) for a standard Gaussian
print(stats.norm.ppf(0.975))              # the 97.5% point: 1.96
print(stats.t.ppf(0.975, df=5))           # wider with few values
print(stats.chi2.sf(12.5, df=4))          # chance of chi^2 >= 12.5 with 4 degrees of freedom
a, b = rng.normal(0, 1, 20), rng.normal(0.5, 1, 20)
print(stats.ttest_ind(a, b, equal_var=False).pvalue, stats.mannwhitneyu(a, b).pvalue)

## <span style="color:#B5121B">4. pandas: one row per group</span>

`groupby` splits a table by a column, applies a summary to each piece, and returns one row per group. This is how you will build a table of results with one row per person, experiment or object.

In [ ]:
import pandas as pd
df = pd.DataFrame({"group": ["a", "a", "b", "b", "b"], "value": [1.0, 1.2, 2.0, 2.4, 2.2]})
summary = df.groupby("group")["value"].agg(n="count", mean="mean", std="std")
summary["sem"] = summary["std"] / np.sqrt(summary["n"])
summary

In [ ]:
# Your turn: build the same kind of table from a loop, one dictionary per group:
rows = []
# for name, g in df.groupby("group"): rows.append(dict(group=name, ...))
# pd.DataFrame(rows)

The loop form is the one to use when each row needs a long calculation (a fit, a spectrum): write a function that takes one group and returns a dictionary, then loop.

## <span style="color:#FD9029">5. Optional practice</span>

Three weather stations logged the temperature every hour for a day. The logger writes **−999** when a reading is missing. Each exercise uses one tool from above; each has a **Check** so you know when you've got it.

In [ ]:
rng_p = np.random.default_rng(7101)
hour = np.tile(np.arange(24), 3)
station = np.repeat(["Quay", "Campus", "Hill"], 24)
base = np.repeat([10.0, 9.0, 7.0], 24); amp = np.repeat([3.0, 3.0, 4.0], 24)
temp = np.round(base + amp * np.sin(2 * np.pi * (hour - 9) / 24) + 0.3 * rng_p.standard_normal(72), 1)
temp[[5, 30, 31, 60]] = -999.0          # missing readings
temp[44] += 6                           # one faulty reading
weather = pd.DataFrame({"station": station, "hour": hour, "temp_C": temp})
weather.head()

**P1 (masks).** How many readings are missing? Replace them with `np.nan`, and find the mean temperature of all the real readings (`np.nanmean`). **Check:** 4 missing; mean 8.81 °C.

In [ ]:
# P1

**P2 (slices).** For the Campus station only, find the hours where the temperature changes by more than 3 °C from the hour before. Why does one faulty reading give *two* jumps? **Check:** hours 20 and 21.

In [ ]:
# P2

**P3 (random numbers).** Simulate 100 000 throws of two dice with one seeded generator (`rng.integers(1, 7, ...)`), and estimate the chance that they add up to 7. Compare with the exact 1/6, then change the seed: how much does the estimate move? **Check:** within about 0.002 of 0.1667.

In [ ]:
# P3

**P4 (scipy.stats).** Ten students measured the same resistor (Ω): `R = np.array([99.6, 100.4, 100.1, 99.8, 100.9, 99.5, 100.2, 100.0, 100.6, 99.9])`. Give the mean, its standard error, and a 95% interval using the *t* distribution (`stats.t.ppf(0.975, df=9)`). How much wider is it than using 1.96? **Check:** 100.10 ± 0.14 Ω; 95% interval 99.79 to 100.41 Ω.

In [ ]:
# P4

**P5 (pandas).** After P1, make a table with one row per station: the number of real readings, the mean, the maximum, and the number missing. Sort it by the mean. **Check:** Hill 6.90, Campus 9.48, Quay 10.07 °C; Campus has 2 missing.

In [ ]:
# P5

**P6 (a function, then a loop).** Write `summarise(values)` that takes one station's temperatures and returns a dictionary (number of real readings, mean, standard error, range). Loop over the stations to build the same kind of table. This is the pattern you will use in Exercise 2: one function for one case, then a loop. **Check:** Quay's range is 6.3 °C (13.3 − 7.0).

In [ ]:
# P6
def summarise(values):
    ...

rows = []
# for name, g in weather.groupby("station"): ...

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>